# Theme #1 + Theme #4: Multi-Agent Negotiation with Self-Improvement

**Hackathon**: OpenEnv India 2026

**Goal**: Train three LLM agents to negotiate resource allocation through emergent cooperation, measured via adaptive curriculum and self-play.

## What This Notebook Does

1. **Theme #1 - Multi-Agent Interactions**: Agents negotiate for shared CPU/GPU/memory under deadlines
2. **Theme #4 - Self-Improvement**: Training improves via curriculum (easy→hard) and self-play evaluation
3. **Generate Training Data**: Environment trajectories → prompt-completion pairs
4. **Train & Evaluate**: Curriculum phases with holdout evaluation
5. **Visualize**: Reward curves, metrics dashboards, and learning progress

**Runtime**: ~20 min on Colab GPU (RL mode) | ~10 min CPU (lightweight)

## Step 1: Setup & Imports

In [ ]:
# Install dependencies
import subprocess
import sys

packages = ['numpy', 'matplotlib', 'requests', 'pydantic', 'openenv-core>=0.1.13']
for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', pkg])

print('✅ Dependencies installed')

In [ ]:
# Clone repo to Colab (or mount Drive if local)
import os
import subprocess

REPO_PATH = '/content/multi-agent'

# Check if already cloned
if not os.path.exists(REPO_PATH):
    print('Cloning repository...')
    subprocess.run(['git', 'clone', 'https://github.com/YOUR_GITHUB/multi-agent.git', REPO_PATH], check=True)
else:
    print('✅ Repository already present')

os.chdir(REPO_PATH)
print(f'Working directory: {os.getcwd()}')

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add repo to path
sys.path.insert(0, os.getcwd())

# Import project modules
from satya_env.env import RealEnvironment
from satya_env.rl_environment import RLFriendlyEnvironment
from src.rl_agent import RLDataLoaderAgent, RLDataCleanerAgent, RLMLTrainerAgent
from src.evaluate import MetricsCalculator, LearningAnalyzer
from src.visualize import ResultsVisualizer

print('✅ All imports successful')
print(f'   Python: {sys.version.split()[0]}')
print(f'   Working dir: {os.getcwd()}')

## Step 2: Configure Training

In [ ]:
# Training configuration
CONFIG = {
    'agent_mode': 'rl',  # Options: 'rl', 'llm', 'hybrid'
    'num_episodes': 30,
    'curriculum_phase_episodes': 5,
    'negotiation_enabled': True,
    'crisis_mode_enabled': True,
    'seed': 42,
    'use_real_env': True,  # Use RLFriendlyEnvironment wrapper
    'save_q_tables': True,
    'generate_plots': True,
}

print('⚙️  Training Configuration:')
for k, v in CONFIG.items():
    print(f'   {k}: {v}')

# Set environment variables
os.environ['TRAINING_AGENT_MODE'] = CONFIG['agent_mode']
os.environ['NUM_EPISODES'] = str(CONFIG['num_episodes'])
os.environ['CURRICULUM_PHASE_EPISODES'] = str(CONFIG['curriculum_phase_episodes'])
os.environ['NEGOTIATION_ENABLED'] = str(CONFIG['negotiation_enabled']).lower()
os.environ['CRISIS_MODE_ENABLED'] = str(CONFIG['crisis_mode_enabled']).lower()
os.environ['SELF_IMPROVEMENT_ENABLED'] = 'true'

## Step 3: Run Training Loop

In [ ]:
# Import train script
import train

print('\n' + '='*70)
print('🚀 STARTING TRAINING LOOP')
print('='*70)

# Run training with imported main function
train.train_agents(num_episodes=CONFIG['num_episodes'])

print('\n' + '='*70)
print('✅ TRAINING COMPLETE')
print('='*70)

## Step 4: Load & Review Results

In [ ]:
import json
from pathlib import Path

results_dir = Path('results')
print('📊 Generated Artifacts:')
print()

artifact_files = [
    ('training_results.json', 'Episode trajectories with rewards'),
    ('episode_metrics.json', 'Per-episode metrics (completion, fairness, etc)'),
    ('selfplay_report.json', 'Theme #4: Curriculum & league duel results'),
    ('holdout_evaluation.json', 'Generalization test on unseen scenarios'),
    ('baseline_comparison.json', 'Trained vs fresh agent comparison'),
    ('negotiation_trace.json', 'Negotiation protocol logs'),
    ('reward_curve.png', 'Reward progression plot'),
    ('metrics_dashboard.png', 'Performance metrics grid'),
]

for fname, desc in artifact_files:
    fpath = results_dir / fname
    if fpath.exists():
        size_kb = fpath.stat().st_size / 1024
        print(f'✅ {fname:35} ({size_kb:6.1f} KB) - {desc}')
    else:
        print(f'⏳ {fname:35} (pending) - {desc}')

print()

In [ ]:
# Load training results
with open('results/training_results.json', 'r') as f:
    all_episodes = json.load(f)

with open('results/episode_metrics.json', 'r') as f:
    metrics = json.load(f)

# Extract key statistics
rewards = [ep['total_reward'] for ep in all_episodes]
completions = [MetricsCalculator.calculate_completion_rate(ep) for ep in all_episodes]
on_time_rates = [MetricsCalculator.calculate_on_time_rate(ep) for ep in all_episodes]

print('📈 TRAINING STATISTICS')
print('='*70)
print(f'Episodes run: {len(all_episodes)}')
print(f'Total episodes: {len(all_episodes)}')
print()
print('Reward Progression:')
print(f'  First 5 avg:  {np.mean(rewards[:5]):7.1f} pts')
print(f'  Last 5 avg:   {np.mean(rewards[-5:]):7.1f} pts')
print(f'  Total change: {np.mean(rewards[-5:]) - np.mean(rewards[:5]):+7.1f} pts')
print(f'  Min/Max:      {min(rewards):7.1f} / {max(rewards):7.1f} pts')
print()
print('Task Completion:')
print(f'  First 5 avg:  {np.mean(completions[:5]):6.1f}%')
print(f'  Last 5 avg:   {np.mean(completions[-5:]):6.1f}%')
print(f'  Change:       {np.mean(completions[-5:]) - np.mean(completions[:5]):+6.1f}%')
print()
print('On-Time Delivery:')
print(f'  First 5 avg:  {np.mean(on_time_rates[:5]):6.1f}%')
print(f'  Last 5 avg:   {np.mean(on_time_rates[-5:]):6.1f}%')
print(f'  Change:       {np.mean(on_time_rates[-5:]) - np.mean(on_time_rates[:5]):+6.1f}%')
print('='*70)

## Step 5: Theme #4 Analysis - Self-Improvement

In [ ]:
# Load Theme #4 results
with open('results/theme4_summary.json', 'r') as f:
    theme4 = json.load(f)

print('🧠 THEME #4: SELF-IMPROVEMENT ANALYSIS')
print('='*70)
print()
print('Curriculum Progression:')
curr_hist = theme4.get('curriculum_history', [])
for entry in curr_hist:
    print(f"  Phase {entry['phase']:2d} → Level {entry['level']} | "
          f"Completion: {entry['completion']:.2f} | "
          f"On-time: {entry['on_time']:.2f} | "
          f"Promoted: {'✅' if entry['promoted'] else '❌'}")

print()
print('Self-Play League Duels:')
duels = theme4.get('league_duels', [])
if duels:
    for duel in duels[:5]:  # Show first 5
        print(f"  Phase {duel['phase']} | Learner: {duel['learner_agent']:20} | "
              f"Reward: {duel['reward']:7.1f} | "
              f"Completion: {duel['completion_rate']:.2f}")
else:
    print("  (No duels run in this training)")

print()
print('Improvement Deltas (Trained vs Fresh):') 
deltas = theme4.get('holdout_delta', {})
print(f"  Reward:            {deltas.get('avg_total_reward', 0):+7.2f}")
print(f"  Completion rate:   {deltas.get('avg_completion_rate', 0):+7.4f}")
print(f"  On-time rate:      {deltas.get('avg_on_time_rate', 0):+7.4f}")
print(f"  Fairness score:    {deltas.get('avg_fairness_score', 0):+7.4f}")
print(f"  Belief accuracy:   {deltas.get('avg_belief_accuracy', 0):+7.4f}")
print('='*70)

## Step 6: Visualizations

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Reward Curve
try:
    img1 = Image.open('results/reward_curve.png')
    axes[0].imshow(img1)
    axes[0].set_title('Reward Progression (Training)', fontsize=14, fontweight='bold')
    axes[0].axis('off')
except:
    axes[0].text(0.5, 0.5, 'Reward curve not found', ha='center')

# Plot 2: Metrics Dashboard
try:
    img2 = Image.open('results/metrics_dashboard.png')
    axes[1].imshow(img2)
    axes[1].set_title('Performance Metrics Dashboard', fontsize=14, fontweight='bold')
    axes[1].axis('off')
except:
    axes[1].text(0.5, 0.5, 'Metrics dashboard not found', ha='center')

plt.tight_layout()
plt.savefig('/tmp/training_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ Plots displayed')

## Step 7: Submission Artifacts

In [ ]:
import json
from pathlib import Path

print('📦 SUBMISSION CHECKLIST')
print('='*70)

checklist = [
    ('✅', 'OpenEnv environment working (satya_env/env.py)'),
    ('✅', 'Theme #1: Multi-agent negotiation functional'),
    ('✅', 'Theme #4: Curriculum + self-play implemented'),
    ('✅', 'RL training loop with Q-learning'),
    ('✅', 'Training results generated (JSON + plots)'),
    ('✅', 'Colab notebook (this file)'),
    ('⏳', 'HuggingFace Space deployment (manual at end)'),
    ('⏳', 'Mini-blog / demo video (to create separately)'),
]

for status, item in checklist:
    print(f'{status}  {item}')

print()
print('📊 Key Metrics for Judges:')
print(f'   Reward improvement: {(np.mean(rewards[-5:]) - np.mean(rewards[:5])):+.1f} pts')
print(f'   Task completion gain: {(np.mean(completions[-5:]) - np.mean(completions[:5])):+.1f}%')
print(f'   On-time delivery gain: {(np.mean(on_time_rates[-5:]) - np.mean(on_time_rates[:5])):+.1f}%')
print()
print('📁 Results location: /content/multi-agent/results/')
print('   - reward_curve.png (for README)')
print('   - metrics_dashboard.png (for README)')
print('   - theme4_summary.json (self-improvement proof)')
print('   - holdout_evaluation.json (generalization test)')
print('='*70)

## Cleanup & Export

In [ ]:
# Download key artifacts
from google.colab import files
import os

print('📥 Ready to download artifacts')
print('   (Uncomment below to download)')
print()
print('# To download training results:')
print('# files.download("results/training_results.json")')
print('# files.download("results/reward_curve.png")')
print('# files.download("results/metrics_dashboard.png")')
print('# files.download("results/theme4_summary.json")')
print()
print('✅ Notebook complete!')